# Seat Occupancy Detection System - Google Colab

A YOLOv5-based system for detecting seat occupancy in images and videos.

## Features
- Upload images or videos for analysis
- Three calibration methods: Upload JSON, Auto-detect chairs, Interactive drawing
- Real-time visualization with matplotlib
- Download annotated results

**Based on**: "Hybrid Multi-Sensor Approach to Intelligent Seat Monitoring" (Dheerendra Pratap et al., Sharda University)

## Section 1: Setup

In [ ]:
# Cell 1.1: Check GPU availability
!nvidia-smi

In [ ]:
# Cell 1.2: Install dependencies
!pip install torch torchvision --quiet
!pip install ultralytics opencv-python-headless matplotlib ipywidgets --quiet

# Verify installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 1.3: Import libraries
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon, Rectangle
from matplotlib.collections import PatchCollection
import json
import os
from google.colab import files
from google.colab import output
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from typing import List, Tuple, Dict, Optional
import time

# Enable interactive widgets
output.enable_custom_widget_manager()

print("All libraries imported successfully!")

In [ ]:
# Cell 1.4: Optional - Mount Google Drive for file persistence
# Uncomment the lines below if you want to save/load files from Google Drive

# from google.colab import drive
# drive.mount('/content/drive')
# print("Google Drive mounted at /content/drive")

## Section 2: Configuration

In [ ]:
# Cell 2.1: Configuration class
class Config:
    """Configuration for Colab seat occupancy detection"""
    
    # Model settings
    MODEL_NAME = 'yolov5s'  # Options: yolov5s, yolov5m, yolov5l
    CONFIDENCE_THRESHOLD = 0.5
    IOU_THRESHOLD = 0.45
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Chair detection settings
    ENABLE_CHAIR_DETECTION = True
    CHAIR_CONFIDENCE_THRESHOLD = 0.4
    
    # Detection parameters
    MIN_OVERLAP_RATIO = 0.3  # 30% overlap to consider seat occupied
    
    # Auto-calibration settings
    AUTO_CALIB_CONFIDENCE = 0.4
    AUTO_CALIB_FRAMES = 30  # For video
    AUTO_CALIB_STABILITY = 15
    AUTO_CALIB_PADDING = 10
    AUTO_CALIB_IOU_THRESHOLD = 0.3
    
    # Visualization colors (RGB for matplotlib)
    COLOR_EMPTY = (0, 1, 0)       # Green
    COLOR_OCCUPIED = (1, 0, 0)    # Red
    COLOR_BBOX = (1, 1, 0)        # Yellow (person)
    COLOR_CHAIR = (1, 0.65, 0)    # Orange (chair)
    COLOR_ZONE_BORDER = (0, 0.5, 1)  # Blue

config = Config()
print(f"Configuration loaded. Using device: {config.DEVICE}")

## Section 3: Core Classes

In [ ]:
# Cell 3.1: Utility functions

def calculate_iou(box1: Tuple[int, int, int, int], box2: Tuple[int, int, int, int]) -> float:
    """
    Calculate Intersection over Union (IoU) between two bounding boxes
    Args: box1, box2: Bounding boxes in format (x1, y1, x2, y2)
    Returns: IoU value between 0 and 1
    """
    x1_1, y1_1, x2_1, y2_1 = box1
    x1_2, y1_2, x2_2, y2_2 = box2

    x_left = max(x1_1, x1_2)
    y_top = max(y1_1, y1_2)
    x_right = min(x2_1, x2_2)
    y_bottom = min(y2_1, y2_2)

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    box1_area = (x2_1 - x1_1) * (y2_1 - y1_1)
    box2_area = (x2_2 - x1_2) * (y2_2 - y1_2)
    union_area = box1_area + box2_area - intersection_area

    return intersection_area / union_area if union_area > 0 else 0.0


def point_in_polygon(point: Tuple[int, int], polygon: List[Tuple[int, int]]) -> bool:
    """
    Check if a point is inside a polygon using ray casting algorithm
    """
    x, y = point
    n = len(polygon)
    inside = False

    p1x, p1y = polygon[0]
    for i in range(1, n + 1):
        p2x, p2y = polygon[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        p1x, p1y = p2x, p2y

    return inside


def box_in_zone(box: Tuple[int, int, int, int], zone: List[Tuple[int, int]],
                min_overlap: float = 0.3) -> bool:
    """
    Check if a bounding box overlaps with a seat zone
    """
    x1, y1, x2, y2 = box
    center_x = (x1 + x2) // 2
    center_y = (y1 + y2) // 2

    if point_in_polygon((center_x, center_y), zone):
        return True

    corners = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
    in_zone_count = sum(1 for corner in corners if point_in_polygon(corner, zone))

    return (in_zone_count / 4) >= min_overlap


def save_zones(zones: Dict, filepath: str = 'seat_zones.json') -> None:
    """Save seat zones to JSON file"""
    # Convert tuples to lists for JSON serialization
    json_zones = {k: [list(p) for p in v] for k, v in zones.items()}
    with open(filepath, 'w') as f:
        json.dump(json_zones, f, indent=4)
    print(f"Seat zones saved to {filepath}")


def load_zones(filepath: str = 'seat_zones.json') -> Dict:
    """Load seat zones from JSON file"""
    if not os.path.exists(filepath):
        return {}
    with open(filepath, 'r') as f:
        zones = json.load(f)
    converted_zones = {}
    for seat_name, points in zones.items():
        converted_zones[seat_name] = [tuple(p) for p in points]
    return converted_zones


def draw_polygon_matplotlib(ax, points, color, linewidth=2, fill=False, alpha=0.3):
    """Draw polygon on matplotlib axis"""
    polygon = MplPolygon(points, fill=fill, 
                         facecolor=color if fill else 'none',
                         edgecolor=color, linewidth=linewidth, 
                         alpha=alpha if fill else 1.0)
    ax.add_patch(polygon)
    return polygon


print("Utility functions loaded!")

In [ ]:
# Cell 3.2: SeatMapper class

class SeatMapper:
    """Maps detected persons to seat zones"""
    
    def __init__(self, seat_zones: Dict[str, List[Tuple[int, int]]], min_overlap: float = 0.3):
        self.seat_zones = seat_zones
        self.min_overlap = min_overlap
        self.occupancy_state = {seat: False for seat in seat_zones.keys()}

    def update_occupancy(self, detections: List[Tuple[int, int, int, int, float]]) -> Dict[str, bool]:
        """Update seat occupancy based on person detections"""
        self.occupancy_state = {seat: False for seat in self.seat_zones.keys()}

        for detection in detections:
            x1, y1, x2, y2, conf = detection
            bbox = (int(x1), int(y1), int(x2), int(y2))

            for seat_name, zone in self.seat_zones.items():
                if box_in_zone(bbox, zone, self.min_overlap):
                    self.occupancy_state[seat_name] = True
                    break

        return self.occupancy_state

    def get_occupancy_stats(self) -> Tuple[int, int, float]:
        """Get (occupied_count, total_count, occupancy_rate)"""
        total = len(self.occupancy_state)
        occupied = sum(1 for is_occupied in self.occupancy_state.values() if is_occupied)
        rate = (occupied / total * 100) if total > 0 else 0.0
        return occupied, total, rate

    def get_empty_seats(self) -> List[str]:
        return [seat for seat, is_occupied in self.occupancy_state.items() if not is_occupied]

    def get_occupied_seats(self) -> List[str]:
        return [seat for seat, is_occupied in self.occupancy_state.items() if is_occupied]

    def is_seat_occupied(self, seat_name: str) -> bool:
        return self.occupancy_state.get(seat_name, False)


print("SeatMapper class loaded!")

In [ ]:
# Cell 3.3: ColabSeatOccupancyDetector class

class ColabSeatOccupancyDetector:
    """Seat occupancy detector adapted for Google Colab"""
    
    def __init__(self, seat_zones: Dict = None):
        """Initialize detector with optional seat zones"""
        self.seat_zones = seat_zones or {}
        self.seat_mapper = SeatMapper(self.seat_zones, config.MIN_OVERLAP_RATIO) if self.seat_zones else None
        
        # Load YOLOv5
        print("Loading YOLOv5 model...")
        self.model = torch.hub.load('ultralytics/yolov5', config.MODEL_NAME, pretrained=True)
        self.model.conf = config.CONFIDENCE_THRESHOLD
        self.model.iou = config.IOU_THRESHOLD
        
        if config.ENABLE_CHAIR_DETECTION:
            self.model.classes = [0, 56]  # persons and chairs
        else:
            self.model.classes = [0]  # only persons
            
        self.model.to(config.DEVICE)
        print(f"Model loaded on {config.DEVICE}")
    
    def detect_objects(self, frame):
        """Detect persons and chairs in frame"""
        results = self.model(frame)
        persons = []
        chairs = []
        
        for *box, conf, cls in results.xyxy[0].cpu().numpy():
            if int(cls) == 0:  # Person
                persons.append((*box, conf))
            elif int(cls) == 56:  # Chair
                if conf >= config.CHAIR_CONFIDENCE_THRESHOLD:
                    chairs.append((*box, conf))
        
        return persons, chairs
    
    def process_frame(self, frame):
        """Process single frame and return data for visualization"""
        persons, chairs = self.detect_objects(frame)
        
        if self.seat_mapper:
            self.seat_mapper.update_occupancy(persons)
            stats = self.seat_mapper.get_occupancy_stats()
        else:
            stats = (0, 0, 0.0)
        
        return frame, persons, chairs, stats
    
    def visualize_frame(self, frame, persons, chairs, figsize=(14, 10)):
        """Visualize frame with matplotlib"""
        fig, ax = plt.subplots(1, 1, figsize=figsize)
        
        # Convert BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ax.imshow(frame_rgb)
        
        # Draw seat zones
        if self.seat_zones:
            for seat_name, zone_points in self.seat_zones.items():
                is_occupied = self.seat_mapper.is_seat_occupied(seat_name) if self.seat_mapper else False
                color = config.COLOR_OCCUPIED if is_occupied else config.COLOR_EMPTY
                
                # Fill zone
                draw_polygon_matplotlib(ax, zone_points, color, fill=True, alpha=0.2)
                # Draw border
                draw_polygon_matplotlib(ax, zone_points, color, linewidth=3)
                
                # Label
                center_x = np.mean([p[0] for p in zone_points])
                center_y = np.mean([p[1] for p in zone_points])
                status = "OCCUPIED" if is_occupied else "EMPTY"
                ax.text(center_x, center_y, f"{seat_name}\n{status}", 
                       ha='center', va='center', fontsize=10,
                       color='white', fontweight='bold',
                       bbox=dict(boxstyle='round', facecolor=color, alpha=0.7))
        
        # Draw person detections
        for x1, y1, x2, y2, conf in persons:
            rect = Rectangle((x1, y1), x2-x1, y2-y1, 
                             fill=False, color=config.COLOR_BBOX, linewidth=2)
            ax.add_patch(rect)
            ax.text(x1, y1-5, f'Person {conf:.2f}', fontsize=8, color='yellow',
                   bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
        
        # Draw chair detections
        for x1, y1, x2, y2, conf in chairs:
            rect = Rectangle((x1, y1), x2-x1, y2-y1,
                             fill=False, color=config.COLOR_CHAIR, linewidth=2)
            ax.add_patch(rect)
            ax.text(x1, y1-5, f'Chair {conf:.2f}', fontsize=8, color='orange',
                   bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
        
        # Statistics panel
        if self.seat_mapper:
            occupied, total, rate = self.seat_mapper.get_occupancy_stats()
            stats_text = f"Occupied: {occupied}/{total} | Rate: {rate:.1f}%"
            ax.text(10, 30, stats_text, fontsize=12, color='white',
                   bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
        
        ax.axis('off')
        plt.tight_layout()
        return fig
    
    def update_zones(self, new_zones):
        """Update seat zones"""
        self.seat_zones = new_zones
        self.seat_mapper = SeatMapper(self.seat_zones, config.MIN_OVERLAP_RATIO) if self.seat_zones else None


print("ColabSeatOccupancyDetector class loaded!")

## Section 4: Calibration Methods

In [ ]:
# Cell 4.1: Method 1 - Upload existing seat_zones.json

def upload_seat_zones():
    """Upload existing seat_zones.json file"""
    print("Upload your seat_zones.json file:")
    uploaded = files.upload()
    
    if uploaded:
        filename = list(uploaded.keys())[0]
        zones = json.loads(uploaded[filename].decode('utf-8'))
        # Convert to tuples
        converted = {k: [tuple(p) for p in v] for k, v in zones.items()}
        print(f"Loaded {len(converted)} seat zones: {list(converted.keys())}")
        return converted
    return None


# Storage for global seat zones
global_seat_zones = None

print("Method 1: Upload JSON - Ready")

In [ ]:
# Cell 4.2: Method 2 - Auto-Calibration

class ColabAutoCalibrator:
    """Auto chair detection for Colab - works with images or video frames"""
    
    CHAIR_CLASS_ID = 56
    
    def __init__(self, confidence_threshold=0.4):
        self.confidence_threshold = confidence_threshold
        print("Loading YOLOv5 model for auto-calibration...")
        self.model = torch.hub.load('ultralytics/yolov5', config.MODEL_NAME, pretrained=True)
        self.model.conf = confidence_threshold
        self.model.to(config.DEVICE)
        print(f"Model loaded on {config.DEVICE}")
    
    def detect_chairs_in_image(self, image):
        """Detect chairs in single image"""
        results = self.model(image)
        chairs = []
        for *box, conf, cls in results.xyxy[0].cpu().numpy():
            if int(cls) == self.CHAIR_CLASS_ID:
                chairs.append((int(box[0]), int(box[1]), int(box[2]), int(box[3]), float(conf)))
        return chairs
    
    def generate_zones_from_chairs(self, chairs, padding=10):
        """Convert chair detections to seat zones"""
        zones = {}
        for i, (x1, y1, x2, y2, conf) in enumerate(chairs, 1):
            x1, y1 = max(0, x1-padding), max(0, y1-padding)
            x2, y2 = x2+padding, y2+padding
            zones[f"seat_{i}"] = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
        return zones
    
    def auto_calibrate_from_image(self, image):
        """Full calibration pipeline for single image"""
        chairs = self.detect_chairs_in_image(image)
        if not chairs:
            print("No chairs detected. Try lowering confidence threshold.")
            return None
        
        zones = self.generate_zones_from_chairs(chairs, config.AUTO_CALIB_PADDING)
        print(f"Auto-detected {len(zones)} seats from {len(chairs)} chairs")
        return zones
    
    def auto_calibrate_from_video(self, video_path, num_frames=30, stability_threshold=15):
        """Calibration from video with frame aggregation"""
        cap = cv2.VideoCapture(video_path)
        all_detections = []
        
        frame_count = 0
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        sample_interval = max(1, total_frames // num_frames)
        
        print(f"Sampling {num_frames} frames from video...")
        while frame_count < num_frames and cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            chairs = self.detect_chairs_in_image(frame)
            all_detections.append(chairs)
            for _ in range(sample_interval - 1):
                cap.read()
            frame_count += 1
        
        cap.release()
        
        # Cluster detections
        stable_chairs = self._cluster_detections(all_detections, stability_threshold)
        if not stable_chairs:
            return None
            
        return self.generate_zones_from_chairs(stable_chairs, config.AUTO_CALIB_PADDING)
    
    def _cluster_detections(self, all_detections, min_appearances):
        """Cluster chair detections across frames"""
        flat = [d for frame_dets in all_detections for d in frame_dets]
        if not flat:
            return []
        
        clusters = []
        used = set()
        
        for i, det1 in enumerate(flat):
            if i in used:
                continue
            cluster = [det1]
            used.add(i)
            
            for j, det2 in enumerate(flat):
                if j in used or j <= i:
                    continue
                if calculate_iou(det1[:4], det2[:4]) >= config.AUTO_CALIB_IOU_THRESHOLD:
                    cluster.append(det2)
                    used.add(j)
            
            if len(cluster) >= min_appearances:
                avg_box = tuple(int(np.mean([d[k] for d in cluster])) for k in range(4))
                avg_conf = np.mean([d[4] for d in cluster])
                clusters.append((*avg_box, avg_conf))
        
        return clusters


print("Method 2: Auto-Calibration - Ready")

In [ ]:
# Cell 4.3: Method 3 - Interactive Zone Drawing

class InteractiveZoneDrawer:
    """Interactive zone drawing using matplotlib click events"""
    
    def __init__(self, image):
        self.image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        self.zones = {}
        self.current_points = []
        self.current_seat_num = 1
        self.fig = None
        self.ax = None
        self.point_markers = []
        self.lines = []
        
    def start(self):
        """Start interactive drawing session"""
        self.fig, self.ax = plt.subplots(figsize=(14, 10))
        self.ax.imshow(self.image)
        self.ax.set_title(f"Click 4 corners for seat_{self.current_seat_num} | "
                         "Right-click to finish | Close window when done", fontsize=12)
        
        self.fig.canvas.mpl_connect('button_press_event', self._on_click)
        
        plt.show()
        return self.zones
    
    def _on_click(self, event):
        if event.inaxes != self.ax:
            return
            
        if event.button == 1:  # Left click
            x, y = int(event.xdata), int(event.ydata)
            self.current_points.append((x, y))
            
            # Draw point
            marker = self.ax.plot(x, y, 'yo', markersize=10)[0]
            self.point_markers.append(marker)
            
            # Draw line to previous point
            if len(self.current_points) > 1:
                prev = self.current_points[-2]
                line = self.ax.plot([prev[0], x], [prev[1], y], 'y-', linewidth=2)[0]
                self.lines.append(line)
            
            # Auto-complete seat if 4 points
            if len(self.current_points) == 4:
                self._complete_seat()
            
            self.fig.canvas.draw()
            
        elif event.button == 3:  # Right click - skip to next seat
            if len(self.current_points) >= 3:
                self._complete_seat()
            else:
                self._clear_current()
            self.fig.canvas.draw()
    
    def _complete_seat(self):
        """Complete current seat zone"""
        seat_name = f"seat_{self.current_seat_num}"
        self.zones[seat_name] = self.current_points.copy()
        
        # Draw completed zone
        draw_polygon_matplotlib(self.ax, self.current_points, 'green', 
                               fill=True, alpha=0.3, linewidth=3)
        
        # Add label
        center_x = np.mean([p[0] for p in self.current_points])
        center_y = np.mean([p[1] for p in self.current_points])
        self.ax.text(center_x, center_y, seat_name, ha='center', va='center',
                    fontsize=10, color='white', fontweight='bold',
                    bbox=dict(facecolor='green', alpha=0.7))
        
        self._clear_current()
        
        self.current_seat_num += 1
        self.ax.set_title(f"Defined {len(self.zones)} zones | "
                         f"Click 4 corners for seat_{self.current_seat_num}")
        print(f"Completed {seat_name}")
    
    def _clear_current(self):
        """Clear current incomplete points"""
        for marker in self.point_markers:
            marker.remove()
        for line in self.lines:
            line.remove()
        self.point_markers = []
        self.lines = []
        self.current_points = []


print("Method 3: Interactive Drawing - Ready")

In [ ]:
# Cell 4.4: Unified Calibration Interface

def calibrate_seats(image, method='auto'):
    """
    Unified calibration interface
    
    Args:
        image: Input image (BGR format from cv2)
        method: 'upload', 'auto', or 'interactive'
    
    Returns:
        Dictionary of seat zones
    """
    global global_seat_zones
    
    if method == 'upload':
        zones = upload_seat_zones()
    
    elif method == 'auto':
        calibrator = ColabAutoCalibrator()
        zones = calibrator.auto_calibrate_from_image(image)
        if zones:
            # Visualize detected zones
            fig, ax = plt.subplots(figsize=(14, 10))
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            for name, points in zones.items():
                draw_polygon_matplotlib(ax, points, 'yellow', fill=True, alpha=0.3)
                center = np.mean(points, axis=0)
                ax.text(center[0], center[1], name, ha='center', va='center',
                       fontsize=10, color='white', fontweight='bold',
                       bbox=dict(facecolor='orange', alpha=0.7))
            ax.set_title(f"Auto-detected {len(zones)} seat zones")
            ax.axis('off')
            plt.show()
    
    elif method == 'interactive':
        print("Instructions:")
        print("- Left-click to add corner points (4 per seat)")
        print("- After 4 clicks, the seat zone is automatically completed")
        print("- Right-click to skip current seat or complete with 3 points")
        print("- Close the window when done defining all seats")
        drawer = InteractiveZoneDrawer(image)
        zones = drawer.start()
    
    else:
        raise ValueError(f"Unknown method: {method}. Use 'upload', 'auto', or 'interactive'")
    
    if zones:
        global_seat_zones = zones
        print(f"\nCalibration complete! {len(zones)} seat zones defined.")
    
    return zones


print("Unified calibration interface ready!")

## Section 5: Image Processing

In [ ]:
# Cell 5.1: Image upload and processing

def upload_image():
    """Upload and display an image"""
    print("Upload an image file (JPG, PNG):")
    uploaded = files.upload()
    
    if not uploaded:
        print("No file uploaded")
        return None, None
    
    filename = list(uploaded.keys())[0]
    
    # Read image
    nparr = np.frombuffer(uploaded[filename], np.uint8)
    image = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    
    print(f"Loaded image: {filename} ({image.shape[1]}x{image.shape[0]})")
    
    # Show original
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(f"Uploaded: {filename}")
    plt.axis('off')
    plt.show()
    
    return image, filename


# Storage for current image
current_image = None
current_filename = None

print("Image upload function ready!")

In [ ]:
# Cell 5.2: Run detection on image

def detect_and_visualize(image, seat_zones=None):
    """Run detection and visualize results"""
    detector = ColabSeatOccupancyDetector(seat_zones)
    
    # Process
    frame, persons, chairs, stats = detector.process_frame(image)
    
    # Visualize
    fig = detector.visualize_frame(frame, persons, chairs)
    plt.show()
    
    # Print statistics
    print(f"\n{'='*50}")
    print("DETECTION RESULTS")
    print(f"{'='*50}")
    print(f"Persons detected: {len(persons)}")
    print(f"Chairs detected: {len(chairs)}")
    
    if seat_zones:
        occupied, total, rate = stats
        print(f"\nSeat Occupancy:")
        print(f"  Occupied: {occupied}/{total} ({rate:.1f}%)")
        print(f"  Empty: {total - occupied}")
        
        print(f"\nPer-Seat Status:")
        for seat_name in seat_zones:
            status = "OCCUPIED" if detector.seat_mapper.is_seat_occupied(seat_name) else "EMPTY"
            print(f"  {seat_name}: {status}")
    else:
        print("\nNo seat zones configured - showing raw detections only")
    
    print(f"{'='*50}")
    
    return detector, fig


print("Detection function ready!")

## Section 6: Video Processing

In [ ]:
# Cell 6.1: Video upload

def upload_video():
    """Upload video file and get info"""
    print("Upload a video file (MP4, AVI, MOV):")
    uploaded = files.upload()
    
    if not uploaded:
        return None
    
    filename = list(uploaded.keys())[0]
    
    # Save to local path for cv2
    video_path = f"/content/{filename}"
    with open(video_path, 'wb') as f:
        f.write(uploaded[filename])
    
    # Get video info
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps > 0 else 0
    cap.release()
    
    print(f"\nLoaded video: {filename}")
    print(f"  Resolution: {width}x{height}")
    print(f"  FPS: {fps:.2f}")
    print(f"  Duration: {duration:.2f}s ({frame_count} frames)")
    
    return video_path


print("Video upload function ready!")

In [ ]:
# Cell 6.2: Process video

def process_video(video_path, seat_zones=None, sample_rate=5, max_frames=100):
    """
    Process video and show sampled frames with detections
    
    Args:
        video_path: Path to video file
        seat_zones: Optional seat zones dictionary
        sample_rate: Process every Nth frame
        max_frames: Maximum frames to process
    """
    detector = ColabSeatOccupancyDetector(seat_zones)
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print("Error: Could not open video")
        return None, None
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Processing video (sampling every {sample_rate} frames)...")
    
    frame_results = []
    frame_idx = 0
    processed_count = 0
    
    while cap.isOpened() and processed_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_idx % sample_rate == 0:
            _, persons, chairs, stats = detector.process_frame(frame)
            
            frame_results.append({
                'frame_idx': frame_idx,
                'time': frame_idx / fps if fps > 0 else 0,
                'persons': len(persons),
                'chairs': len(chairs),
                'occupied': stats[0],
                'total': stats[1],
                'rate': stats[2]
            })
            
            processed_count += 1
            
            if processed_count % 20 == 0:
                print(f"  Processed {processed_count} frames...")
        
        frame_idx += 1
    
    cap.release()
    
    print(f"\nProcessed {processed_count} frames from {frame_idx} total")
    
    # Plot occupancy over time
    if seat_zones and frame_results:
        times = [r['time'] for r in frame_results]
        rates = [r['rate'] for r in frame_results]
        persons_count = [r['persons'] for r in frame_results]
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
        
        # Occupancy rate
        ax1.plot(times, rates, 'b-', linewidth=2)
        ax1.fill_between(times, rates, alpha=0.3)
        ax1.set_xlabel('Time (seconds)')
        ax1.set_ylabel('Occupancy Rate (%)')
        ax1.set_title('Seat Occupancy Over Time')
        ax1.grid(True, alpha=0.3)
        ax1.set_ylim(0, 100)
        
        # Person count
        ax2.plot(times, persons_count, 'g-', linewidth=2)
        ax2.fill_between(times, persons_count, alpha=0.3, color='green')
        ax2.set_xlabel('Time (seconds)')
        ax2.set_ylabel('Persons Detected')
        ax2.set_title('Person Detection Over Time')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    return frame_results, detector


print("Video processing function ready!")

In [ ]:
# Cell 6.3: Show specific frame from video

def show_video_frame(video_path, frame_number, seat_zones=None):
    """Show a specific frame from video with detection"""
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ret, frame = cap.read()
    cap.release()
    
    if not ret:
        print(f"Could not read frame {frame_number}")
        return
    
    detector = ColabSeatOccupancyDetector(seat_zones)
    _, persons, chairs, _ = detector.process_frame(frame)
    fig = detector.visualize_frame(frame, persons, chairs)
    plt.title(f"Frame {frame_number}")
    plt.show()


print("Frame viewer function ready!")

## Section 7: Interactive Demo

In [ ]:
# Cell 7.1: Complete Demo Workflow

print("="*60)
print("SEAT OCCUPANCY DETECTION - GOOGLE COLAB DEMO")
print("="*60)
print("\nThis interactive demo guides you through the complete workflow.")
print("\nSteps:")
print("1. Upload an image or video")
print("2. Choose a calibration method to define seat zones")
print("3. Run detection and view results")
print("4. Export annotated results if needed")

# Create widgets
input_type = widgets.Dropdown(
    options=[('Single Image', 'image'), ('Video File', 'video')],
    description='Input Type:',
    style={'description_width': '100px'}
)

calib_method = widgets.Dropdown(
    options=[
        ('Upload seat_zones.json', 'upload'),
        ('Auto-detect chairs', 'auto'),
        ('Draw zones interactively', 'interactive'),
        ('Skip (raw detection only)', 'skip')
    ],
    description='Calibration:',
    style={'description_width': '100px'}
)

sample_rate_slider = widgets.IntSlider(
    value=5, min=1, max=30, step=1,
    description='Sample Rate:',
    style={'description_width': '100px'},
    tooltip='Process every Nth frame (for video)'
)

run_button = widgets.Button(
    description='Run Demo',
    button_style='success',
    icon='play'
)

output_area = widgets.Output()

# Store state
demo_state = {
    'image': None,
    'video_path': None,
    'seat_zones': None,
    'detector': None
}

def run_demo(b):
    with output_area:
        clear_output(wait=True)
        
        input_t = input_type.value
        calib_m = calib_method.value
        sample_r = sample_rate_slider.value
        
        print(f"\n{'='*50}")
        print(f"Running demo with:")
        print(f"  Input: {input_t}")
        print(f"  Calibration: {calib_m}")
        print(f"{'='*50}\n")
        
        # Step 1: Get input
        print("Step 1: Upload input file")
        if input_t == 'image':
            image, _ = upload_image()
            if image is None:
                return
            demo_state['image'] = image
        else:
            video_path = upload_video()
            if video_path is None:
                return
            demo_state['video_path'] = video_path
            # Get first frame for calibration
            cap = cv2.VideoCapture(video_path)
            _, image = cap.read()
            cap.release()
            demo_state['image'] = image
        
        # Step 2: Calibrate
        print(f"\nStep 2: Calibration ({calib_m})")
        if calib_m == 'skip':
            seat_zones = None
            print("Skipping calibration - will show raw detections only")
        else:
            seat_zones = calibrate_seats(demo_state['image'], calib_m)
        
        demo_state['seat_zones'] = seat_zones
        
        # Step 3: Run detection
        print(f"\nStep 3: Running detection...")
        if input_t == 'image':
            detector, _ = detect_and_visualize(demo_state['image'], seat_zones)
            demo_state['detector'] = detector
        else:
            results, detector = process_video(
                demo_state['video_path'], 
                seat_zones, 
                sample_rate=sample_r
            )
            demo_state['detector'] = detector
            
            # Show a sample frame
            if results:
                mid_idx = len(results) // 2
                print(f"\nShowing frame at {results[mid_idx]['time']:.2f}s:")
                show_video_frame(
                    demo_state['video_path'],
                    results[mid_idx]['frame_idx'],
                    seat_zones
                )
        
        print("\n" + "="*50)
        print("Demo complete!")
        print("Use the export functions below to save your results.")
        print("="*50)

run_button.on_click(run_demo)

# Display widgets
display(widgets.VBox([
    widgets.HTML("<h3>Demo Configuration</h3>"),
    input_type,
    calib_method,
    sample_rate_slider,
    run_button
]))
display(output_area)

## Section 8: Export & Download Results

In [ ]:
# Cell 8.1: Save and download functions

def save_annotated_image(image, seat_zones=None, output_filename="annotated_result.jpg"):
    """Save annotated image and download"""
    detector = ColabSeatOccupancyDetector(seat_zones)
    frame, persons, chairs, _ = detector.process_frame(image)
    fig = detector.visualize_frame(frame, persons, chairs, figsize=(14, 10))
    
    fig.savefig(output_filename, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    
    files.download(output_filename)
    print(f"Downloaded: {output_filename}")


def save_seat_zones_json(zones, filename="seat_zones_colab.json"):
    """Save seat zones to JSON and download"""
    if not zones:
        print("No seat zones to save!")
        return
    
    # Convert tuples to lists for JSON
    json_zones = {k: [list(p) for p in v] for k, v in zones.items()}
    
    with open(filename, 'w') as f:
        json.dump(json_zones, f, indent=2)
    
    files.download(filename)
    print(f"Downloaded: {filename}")


def export_detection_summary(results, filename="detection_summary.json"):
    """Export video detection results as JSON"""
    if not results:
        print("No results to export!")
        return
    
    with open(filename, 'w') as f:
        json.dump(results, f, indent=2)
    
    files.download(filename)
    print(f"Downloaded: {filename}")


print("Export functions ready!")
print("\nAvailable export functions:")
print("  - save_annotated_image(image, seat_zones) - Download annotated image")
print("  - save_seat_zones_json(zones) - Download seat zones as JSON")
print("  - export_detection_summary(results) - Download video analysis results")

In [ ]:
# Cell 8.2: Quick export buttons (use after running demo)

export_image_btn = widgets.Button(description="Export Annotated Image", button_style='info', icon='download')
export_zones_btn = widgets.Button(description="Export Seat Zones JSON", button_style='info', icon='download')
export_output = widgets.Output()

def on_export_image(b):
    with export_output:
        clear_output()
        if demo_state.get('image') is not None:
            save_annotated_image(demo_state['image'], demo_state.get('seat_zones'))
        else:
            print("No image available. Run the demo first!")

def on_export_zones(b):
    with export_output:
        clear_output()
        if demo_state.get('seat_zones'):
            save_seat_zones_json(demo_state['seat_zones'])
        else:
            print("No seat zones defined. Run calibration first!")

export_image_btn.on_click(on_export_image)
export_zones_btn.on_click(on_export_zones)

display(widgets.HTML("<h3>Quick Export</h3>"))
display(widgets.HBox([export_image_btn, export_zones_btn]))
display(export_output)

---

## Quick Start Guide

### Option 1: Use the Interactive Demo (Section 7)
1. Run all cells in Sections 1-6 to load the system
2. Use the interactive demo in Section 7 to upload files and run detection

### Option 2: Manual Workflow

```python
# 1. Upload an image
image, filename = upload_image()

# 2. Calibrate (choose one method)
seat_zones = calibrate_seats(image, 'auto')      # Auto-detect chairs
# OR
seat_zones = calibrate_seats(image, 'upload')    # Upload existing JSON
# OR
seat_zones = calibrate_seats(image, 'interactive')  # Draw manually

# 3. Run detection
detector, fig = detect_and_visualize(image, seat_zones)

# 4. Export results
save_annotated_image(image, seat_zones)
save_seat_zones_json(seat_zones)
```

### For Video Processing

```python
# 1. Upload video
video_path = upload_video()

# 2. Get first frame for calibration
cap = cv2.VideoCapture(video_path)
_, frame = cap.read()
cap.release()

# 3. Calibrate on first frame
seat_zones = calibrate_seats(frame, 'auto')

# 4. Process video
results, detector = process_video(video_path, seat_zones, sample_rate=5)

# 5. View specific frame
show_video_frame(video_path, 100, seat_zones)  # Show frame 100
```